# PLaMo 3 Translate を Colab で OpenAI 互換 API に

構成: `llama-server`（plamo-3-translate-0.8b-Q4_K_M, :8000）→ `plamo_proxy`（YakuCraft のバッチ要求 → PFML3 で 1 件ずつ `/completion`＋APIキー認証＋記録, :8100）→ Cloudflare Tunnel（https）

1. セルを上から順に実行
2. ④で表示される **接続先URL / APIキー / モデル名** をアプリの「OpenAI 互換（カスタム）」に入力

0.8B なので CPU ランタイムでも動きます。リクエストと返答は `/content/requests.jsonl` に記録されます。

### YakuCraft 1.3.0 での使い方
- 「OpenAI 互換（カスタム）」の **バッチサイズは 10〜20** を推奨（アプリの HTTP タイムアウト 120 秒・Cloudflare の約 100 秒制限に収めるため）。並列数は 1〜2。
- 時間内に終わらなかった項目・書式を壊した項目は訳文を返しません。アプリ側では「応答なし」として未翻訳のまま残るので、再実行すれば拾えます。理由は `requests.jsonl` の `failures` に残ります。


In [ ]:
#@title ① 設定
API_KEY = ""  #@param {type:"string"}
#@markdown 空にするとランダム生成

#@markdown ---
#@markdown ### 実行環境
USE_GPU = False  #@param {type:"boolean"}
USE_DRIVE_LLAMA = True  #@param {type:"boolean"}
DRIVE_LLAMA_DIR = "/content/drive/MyDrive/llama"  #@param {type:"string"}
#@markdown ON: Google ドライブの llama フォルダにある llama-server（llama-server-static / Linux 用 zip）を使う。無い・動かない場合は自動でダウンロード／ビルドに切替
#@markdown ON: CUDA 版 llama.cpp をビルド（GPU ランタイム必須・初回 10〜15 分）／ OFF: CPU 版をダウンロード（すぐ起動）

#@markdown ---
#@markdown ### Context and Offload
CTX_SIZE = 8192  #@param {type:"integer"}
#@markdown コンテキスト長（全スロット合計）。マイクラの lang 1行は短いので 8192 で十分
GPU_LAYERS = 99  #@param {type:"integer"}
#@markdown GPUオフロード層数（USE_GPU=OFF のときは無視。99 = 全部）

#@markdown ---
#@markdown ### Advanced
THREADS = 0  #@param {type:"integer"}
#@markdown CPUスレッド数（0 = 自動）
BATCH_SIZE = 2048  #@param {type:"integer"}
#@markdown 評価バッチサイズ (-b)
UBATCH_SIZE = 512  #@param {type:"integer"}
#@markdown Physical Batch Size (-ub)
PARALLEL = 4  #@param {type:"integer"}
#@markdown Max Concurrent（同時翻訳数。プロキシの並列数も同じ値になります）
KV_UNIFIED = True  #@param {type:"boolean"}
#@markdown Unified KV Cache（ON 推奨: 短い文字列を並列で流すマイクラ翻訳向き）
CTX_CHECKPOINTS = 32  #@param {type:"integer"}
ROPE_FREQ_BASE = 0.0  #@param {type:"number"}
#@markdown 0 = 自動
ROPE_FREQ_SCALE = 0.0  #@param {type:"number"}
#@markdown 0 = 自動
KV_OFFLOAD = True  #@param {type:"boolean"}
#@markdown KVキャッシュをGPUメモリにオフロード
MLOCK = True  #@param {type:"boolean"}
#@markdown モデルをメモリに保持
USE_MMAP = True  #@param {type:"boolean"}
SEED = -1  #@param {type:"integer"}
#@markdown -1 = ランダム（temperature 0 なので結果はほぼ固定）
FLASH_ATTN = "on"  #@param ["on", "off", "auto"]

#@markdown ---
#@markdown ### 翻訳プロキシ（マイクラ用）
MAX_TOKENS = 1024  #@param {type:"integer"}
#@markdown 1エントリあたりの出力トークン上限。実際は「原文トークン数×2+24（最低64）」をこの値で頭打ちにして使う
TEMPERATURE = 0.0  #@param {type:"number"}
REQUEST_TIMEOUT = 120  #@param {type:"integer"}
#@markdown 1エントリあたりのタイムアウト秒
BATCH_DEADLINE = 85  #@param {type:"integer"}
#@markdown 1バッチの持ち時間（秒）。超えた分は返さず次回に回す。Cloudflare Quick Tunnel は約100秒で切れるので 90 未満に

if PARALLEL < 1: PARALLEL = 1
if not KV_UNIFIED and CTX_SIZE // PARALLEL < 1024:
    print(f"⚠ 1スロットあたり {CTX_SIZE // PARALLEL} トークンしかありません。CTX_SIZE を増やすか KV_UNIFIED を ON に")
MODEL_URL = "https://huggingface.co/pfnet/plamo-3-translate-0.8b-GGUF/resolve/main/plamo-3-translate-0.8b-Q4_K_M.gguf?download=true"
MODEL_PATH = "/content/plamo-3-translate-0.8b-Q4_K_M.gguf"
MODEL_NAME = "plamo-3-translate"

import secrets
if not API_KEY:
    API_KEY = "sk-colab-" + secrets.token_urlsafe(24)
print("API_KEY:", API_KEY)

In [ ]:
#@title ② llama.cpp・cloudflared・モデルを取得して llama-server を起動
import json, os, glob, subprocess, time, urllib.request, zipfile, tarfile

import shutil
if USE_GPU:
    if shutil.which("nvidia-smi") is None or subprocess.run(["nvidia-smi"], capture_output=True).returncode != 0:
        print("⚠ GPU が見つかりません（CPU ランタイムです）→ CPU 版で続行します")
        print("   GPU を使うには「ランタイム → ランタイムのタイプを変更」で T4 GPU を選んで再実行")
        USE_GPU = False

# CUDA ランタイムのライブラリパス（CUDA 版バイナリ用）
LIB_DIRS = [d for d in ["/usr/local/cuda/lib64", "/usr/lib64-nvidia", "/usr/local/nvidia/lib64"] if os.path.isdir(d)]

def try_binary(path, extra_dirs=()):
    """実行できるか --version で確認"""
    os.chmod(path, 0o755)
    e = dict(os.environ, LD_LIBRARY_PATH=":".join([os.path.dirname(path), *extra_dirs, *LIB_DIRS]))
    r = subprocess.run([path, "--version"], capture_output=True, text=True, env=e)
    print((r.stdout + r.stderr).strip()[-500:])
    return r.returncode == 0

LLAMA_SERVER = None
if USE_DRIVE_LLAMA:
    from google.colab import drive
    if not os.path.isdir("/content/drive/MyDrive"):
        drive.mount("/content/drive")
    if not os.path.isdir(DRIVE_LLAMA_DIR):
        print("⚠ ドライブに", DRIVE_LLAMA_DIR, "がありません")
    else:
        print("ドライブ:", sorted(os.listdir(DRIVE_LLAMA_DIR)))
        os.makedirs("/content/llama_drive", exist_ok=True)
        # 1) 公式ビルドの zip / tar.gz を展開（cudart-* は CUDA ライブラリとして一緒に展開）
        for name in sorted(os.listdir(DRIVE_LLAMA_DIR)):
            src = os.path.join(DRIVE_LLAMA_DIR, name)
            if "win" in name.lower():
                print("スキップ（Windows 用）:", name); continue
            if "cuda" in name.lower() and not USE_GPU:
                print("スキップ（CUDA 版・USE_GPU=OFF）:", name); continue
            if name.endswith(".zip"):
                zipfile.ZipFile(src).extractall("/content/llama_drive")
            elif name.endswith((".tar.gz", ".tgz")):
                tarfile.open(src).extractall("/content/llama_drive")
        libdirs = {os.path.dirname(p) for p in glob.glob("/content/llama_drive/**/*.so*", recursive=True)}
        for cand in glob.glob("/content/llama_drive/**/llama-server", recursive=True):
            print("試行:", cand)
            if try_binary(cand, libdirs):
                LLAMA_SERVER = cand; LIB_DIRS[:0] = list(libdirs); break
            print("  → 実行できませんでした")
        # 2) llama-server-static（単体バイナリ。対応オプションが少ないことがある）
        if LLAMA_SERVER is None:
            for name in sorted(os.listdir(DRIVE_LLAMA_DIR)):
                if name.startswith("llama-server") and not name.endswith((".zip", ".tar.gz", ".exe")):
                    dst = "/content/llama_drive/" + name
                    shutil.copy(os.path.join(DRIVE_LLAMA_DIR, name), dst)
                    print("試行:", name)
                    if try_binary(dst):
                        LLAMA_SERVER = dst; break
                    print("  → 実行できませんでした")
    if LLAMA_SERVER:
        print("✅ ドライブの llama-server を使用:", LLAMA_SERVER)
    else:
        print("⚠ ドライブに使える Linux 用 llama-server がありません → 通常の方法で取得します")

if USE_GPU and not LLAMA_SERVER:
    # CUDA 版をソースからビルド（2回目以降はスキップ）
    LLAMA_SERVER = "/content/llama.cpp/build/bin/llama-server"
    if not os.path.exists(LLAMA_SERVER):
        cuda_bin = next((d for d in ["/usr/local/cuda/bin", *sorted(glob.glob("/usr/local/cuda-*/bin"), reverse=True)]
                         if os.path.exists(d + "/nvcc")), None)
        if cuda_bin is None:
            raise RuntimeError("nvcc（CUDA Toolkit）が見つかりません。USE_GPU を OFF にして実行してください")
        os.environ["PATH"] = cuda_bin + ":" + os.environ["PATH"]
        os.environ["CUDACXX"] = cuda_bin + "/nvcc"
        os.environ["CUDAToolkit_ROOT"] = os.path.dirname(cuda_bin)
        cap = subprocess.run(["nvidia-smi", "--query-gpu=compute_cap", "--format=csv,noheader"],
                             capture_output=True, text=True).stdout.split()[0].replace(".", "")
        print("CUDA:", cuda_bin, "/ compute capability:", cap)
        shutil.rmtree("/content/llama.cpp", ignore_errors=True)
        !git clone -q --depth 1 https://github.com/ggml-org/llama.cpp /content/llama.cpp
        !cd /content/llama.cpp && cmake -B build -DGGML_CUDA=ON -DCMAKE_CUDA_ARCHITECTURES={cap} -DLLAMA_CURL=OFF -DCMAKE_BUILD_TYPE=Release 2>&1 | tail -n 5 && cmake --build build --target llama-server -j$(nproc) 2>&1 | tail -n 3
        if not os.path.exists(LLAMA_SERVER):
            raise RuntimeError("CUDA ビルドに失敗しました（上のログを確認）。USE_GPU を OFF にすれば CPU 版で動きます")

if not LLAMA_SERVER:
    # llama.cpp の最新 Ubuntu x64 (CPU) ビルドを取得
    rel = json.load(urllib.request.urlopen("https://api.github.com/repos/ggml-org/llama.cpp/releases/latest"))
    asset = next(a for a in rel["assets"] if "bin-ubuntu-x64" in a["name"])
    print("llama.cpp:", rel["tag_name"], asset["name"])
    urllib.request.urlretrieve(asset["browser_download_url"], "/content/llama_bin")
    os.makedirs("/content/llama", exist_ok=True)
    if asset["name"].endswith(".zip"):
        zipfile.ZipFile("/content/llama_bin").extractall("/content/llama")
    else:
        tarfile.open("/content/llama_bin").extractall("/content/llama")
    LLAMA_SERVER = glob.glob("/content/llama/**/llama-server", recursive=True)[0]
os.chmod(LLAMA_SERVER, 0o755)

!wget -q -O /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x /usr/local/bin/cloudflared

if not os.path.exists(MODEL_PATH):
    !wget -q --show-progress -O {MODEL_PATH} "{MODEL_URL}"

env = dict(os.environ, LD_LIBRARY_PATH=":".join([os.path.dirname(LLAMA_SERVER), *LIB_DIRS]))
args = [LLAMA_SERVER, "-m", MODEL_PATH, "--host", "127.0.0.1", "--port", "8000",
        "--alias", MODEL_NAME,
        "--special",  # 特殊トークンを本文に出す → stop 文字列 <|plamo:tag|> が確実に効く
        "-c", str(CTX_SIZE), "-np", str(PARALLEL), "-b", str(BATCH_SIZE), "-ub", str(UBATCH_SIZE),
        "-ngl", str(GPU_LAYERS if USE_GPU else 0), "--seed", str(SEED), "-fa", FLASH_ATTN,
        "--ctx-checkpoints", str(CTX_CHECKPOINTS)]
if THREADS > 0:         args += ["-t", str(THREADS)]
if KV_UNIFIED:          args += ["--kv-unified"]
if ROPE_FREQ_BASE > 0:  args += ["--rope-freq-base", str(ROPE_FREQ_BASE)]
if ROPE_FREQ_SCALE > 0: args += ["--rope-freq-scale", str(ROPE_FREQ_SCALE)]
if not KV_OFFLOAD:      args += ["--no-kv-offload"]
if MLOCK:               args += ["--mlock"]
if not USE_MMAP:        args += ["--no-mmap"]
# このビルドが対応していないオプションは外す（例: 古い/派生ビルドの --mlock など）
import re
help_txt = subprocess.run([LLAMA_SERVER, "--help"], capture_output=True, text=True, env=env)
help_txt = help_txt.stdout + help_txt.stderr
REQUIRED = {"-m", "--host", "--port"}
kept, i = [args[0]], 1
while i < len(args):
    opt = args[i]
    has_val = i + 1 < len(args) and not args[i + 1].startswith("--") and not (args[i + 1].startswith("-") and not args[i + 1].lstrip("-").replace(".", "").isdigit())
    chunk = args[i:i + 2] if has_val else args[i:i + 1]
    if opt in REQUIRED or re.search(r"(^|[\s,])" + re.escape(opt) + r"([\s,=]|$)", help_txt, re.M):
        kept += chunk
    else:
        print("⚠ 未対応のため無視:", " ".join(chunk))
    i += len(chunk)
args = kept
print("起動:", " ".join(args[1:]))
if 'llama' in globals() and llama.poll() is None:  # 設定を変えて再実行したとき
    llama.terminate(); llama.wait()
llama = subprocess.Popen(args, env=env, stdout=open("/content/llama.log", "w"), stderr=subprocess.STDOUT)
for _ in range(120):
    try:
        if json.load(urllib.request.urlopen("http://127.0.0.1:8000/health")).get("status") == "ok":
            break
    except Exception:
        pass
    if llama.poll() is not None:
        raise RuntimeError("llama-server が停止しました:\n" + open("/content/llama.log").read()[-3000:])
    time.sleep(1)
print("✅ llama-server 起動 (:8000)")

In [ ]:
%%writefile /content/plamo_proxy.py
"""YakuCraft 1.3.0 の「OpenAI 互換（カスタム）」バッチ要求を PLaMo 3 Translate 向けに変換するプロキシ。

アプリ → POST /v1/chat/completions
  messages[-1].content = {"mod_name","namespace","glossary":[{source,target}],
                          "entries":[{id,key,source,source_locale,target_locale,entry_kind,placeholders}]}
  response_format = json_schema（無視してよい。返す形は同じ）
アプリ ← choices[0].message.content = {"translations":[{"id":1,"translation":"..."}]}

PLaMo 側は YakuCraft 本体の PLaMo Provider（yc-plamo）と同じ方式で呼ぶ:
  - PFML3 プロンプトを自前で組み立てて llama-server の /completion へ 1 件ずつ送る
  - placeholder・書式コードはマーカーへ退避し、復元後に原文と厳密照合する
  - 失敗した項目は訳文を返さない（原文を訳文として返すと「翻訳済み」として取り込まれるため）
標準ライブラリのみ。
"""
import collections
import datetime
import json
import os
import re
import sys
import threading
import time
import urllib.error
import urllib.request
from concurrent.futures import ThreadPoolExecutor, wait
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer

LLAMA_URL = os.environ.get("LLAMA_URL", "http://127.0.0.1:8000")
PLAMO_MODEL = os.environ.get("MODEL_NAME", "plamo-3-translate")
LISTEN_PORT = int(os.environ.get("LISTEN_PORT", "8100"))
PARALLEL = int(os.environ.get("PARALLEL", "4"))  # llama-server の -np と合わせる
API_KEY = os.environ.get("API_KEY", "")
MAX_TOKENS = int(os.environ.get("MAX_TOKENS", "1024"))
TEMPERATURE = float(os.environ.get("TEMPERATURE", "0"))
TIMEOUT = int(os.environ.get("REQUEST_TIMEOUT", "120"))
# 1 バッチの持ち時間。アプリの HTTP タイムアウトは 120 秒、Cloudflare Quick Tunnel は
# 約 100 秒で 524 を返す。これを超えるとアプリが同じバッチを丸ごと再送するので、
# 時間内に終わった分だけ返し、残りは「応答なし」として次回に回す。
BATCH_DEADLINE = float(os.environ.get("BATCH_DEADLINE", "85"))
LOG_FILE = os.environ.get("LOG_FILE", "/content/requests.jsonl")

MIN_OUTPUT_TOKENS = 64
STOP_STRINGS = ["<|plamo:tag|>", "<|plamo:bos|>"]
TAG, MSG, KEY, VAL = "<|plamo:tag|>", "<|plamo:msg|>", "<|plamo:key|>", "<|plamo:val|>"
INDEX, ITEM = "<|plamo:index|>", "<|plamo:item|>"

# ---------------------------------------------------------------- 言語コード
PLAMO_LANGUAGES = {
    "ja", "en", "zh-hans", "zh-hant", "ko", "ar", "az", "be", "bg", "bn", "ca", "ceb", "de", "es",
    "et", "eu", "fa", "fr", "gu", "he", "hi", "hr", "hu", "hy", "id", "it", "jv", "ka", "kk", "km",
    "lt", "lv", "ml", "mr", "ms", "my", "ne", "nl", "pa", "pl", "ru", "si", "sk", "sl", "sr", "sw",
    "ta", "te", "th", "tl", "ur", "uz", "vi",
}
LOCALE_OVERRIDES = {
    "zh_cn": "zh-hans", "zh_hans": "zh-hans",
    "zh_tw": "zh-hant", "zh_hk": "zh-hant", "zh_hant": "zh-hant",
    "de_at": None, "de_ch": None,  # 方言は標準語で代用しない
    "val_es": "ca", "fil_ph": "tl", "zlm_arab": "ms",
}
# 人工言語は先頭一致（en_pt → en など）より先に弾く
CONSTRUCTED = {"en_ud", "en_pt", "enp", "enws", "lol_us", "tok", "tlh_aa", "qya_aa", "isv", "ldn"}


def plamo_lang(locale):
    loc = (locale or "").strip().replace("-", "_").lower()
    if not loc or loc in CONSTRUCTED:
        return None
    if loc in LOCALE_OVERRIDES:
        return LOCALE_OVERRIDES[loc]
    lang = loc.split("_")[0]
    return lang if lang in PLAMO_LANGUAGES else None


# ---------------------------------------------------------------- 特殊トークン対策
def unsafe(text):
    """llama-server はプロンプト中の特殊トークン表記を本物として解釈するので、混入を拒否する。"""
    if "<|plamo:" in text:
        return True
    i = text.find("<|")
    return i >= 0 and text.find("|>", i + 2) >= 0


# ---------------------------------------------------------------- 保護区間（placeholder・書式コード）
COMMON_PATTERNS = [
    r"§x(?:§[0-9a-fA-F]){6}",                              # 16進カラー
    r"§[0-9a-fk-orA-FK-OR]",                                # Minecraft 書式コード
    r"%(?:\d+\$)?[-#+0,(]*\d*(?:\.\d+)?[sdfxXcbeEgGoh%]",   # printf
    r"\{\d+\}",                                             # {0}
    r"\\n",                                                 # 文字列としての \n
    r"\r?\n",                                               # 本物の改行
]
PROFILE_PATTERNS = {
    "lang": [],
    "patchouli": [r"\$\([^()]*\)", r"#[A-Za-z0-9_.:\-]+#"],
    "ftbquests": [r"&[0-9a-fk-orA-FK-OR]", r"\{[^{}\n]*\}"],
}


def span_regex(entry):
    """アプリが送ってくる placeholders（yc-validation の抽出結果）と既定パターンの和。"""
    literals = sorted({p for p in entry.get("placeholders") or [] if isinstance(p, str) and p},
                      key=len, reverse=True)
    parts = [re.escape(p) for p in literals]
    parts += PROFILE_PATTERNS.get(entry.get("entry_kind") or "lang", []) + COMMON_PATTERNS
    return re.compile("|".join(f"(?:{p})" for p in parts))


def is_open(raw):
    return raw.startswith("$(") and raw not in ("$()", "$(br)", "$(br2)") and not raw.startswith("$(k:")


def pair_roles(spans):
    """Patchouli の $(item)…$() を開き／閉じの組にする。"""
    roles, openers, stack = ["single"] * len(spans), [None] * len(spans), []
    for i, raw in enumerate(spans):
        if raw == "$()":
            if stack:
                o = stack.pop()
                roles[o], roles[i], openers[i] = "open", "close", o
        elif is_open(raw):
            stack.append(i)
    return roles, openers


def marker(strategy, i, role):
    if strategy == "bracket":
        return f"⟦{i}⟧"
    if strategy == "ascii":
        return f"__YC_PH_{i:04d}__"
    return {"open": f"<x{i}>", "close": f"</x{i}>", "single": f"<x{i}/>"}[role]


MARKER_RE = {
    "bracket": re.compile(r"⟦(\d+)⟧"),
    "ascii": re.compile(r"__YC_PH_(\d+)__"),
    "paired": re.compile(r"<(/?)x(\d+)(/?)>"),
}
MARKER_FRAGMENTS = {"bracket": ("⟦", "⟧"), "ascii": ("__YC_PH_",), "paired": ("<x", "</x")}


class Mismatch(Exception):
    pass


def mask(source, regex, strategy):
    spans = [m.group(0) for m in regex.finditer(source)]
    roles, openers = pair_roles(spans)
    it = iter(range(len(spans)))
    masked = regex.sub(lambda m: marker(strategy, (i := next(it)), roles[i]), source)
    return masked, spans, roles, openers


def restore(output, source, regex, strategy, spans, roles, openers):
    seen = []

    def rep(m):
        if strategy == "paired":
            close, idx, single = m.group(1) == "/", int(m.group(2)), m.group(3) == "/"
            role = {(False, False): "open", (True, False): "close", (False, True): "single"}.get((close, single))
            if idx >= len(spans) or role != roles[idx]:
                raise Mismatch("role")
        else:
            idx = int(m.group(1))
            if idx >= len(spans):
                raise Mismatch("unknown marker")
        if m.group(0) != marker(strategy, idx, roles[idx]):
            raise Mismatch("non-canonical marker")
        seen.append(idx)
        return "\x00%d\x00" % idx  # 一旦退避（復元した生テキストを再走査しないため）

    body = MARKER_RE[strategy].sub(rep, output.strip())
    if sorted(seen) != list(range(len(spans))):
        raise Mismatch("missing/duplicated marker")
    # 開き → 閉じの順序が保たれているか
    stack = []
    for idx in seen:
        if roles[idx] == "open":
            stack.append(idx)
        elif roles[idx] == "close" and (not stack or stack.pop() != openers[idx]):
            raise Mismatch("pair order")
    # 書き損じたマーカーが本文に残っていないか（原文に元からある分は許す）
    for frag in MARKER_FRAGMENTS[strategy]:
        if body.count(frag) > source.count(frag):
            raise Mismatch("broken marker")
    body = re.sub("\x00(\\d+)\x00", lambda m: spans[int(m.group(1))], body).strip()
    lead = source[: len(source) - len(source.lstrip())]
    trail = source[len(source.rstrip()):]
    restored = lead + body + trail
    if collections.Counter(m.group(0) for m in regex.finditer(restored)) != collections.Counter(spans):
        raise Mismatch("span multiset")
    return restored


def strategies_for(entry):
    # yc-plamo の実測: Patchouli は開き/閉じタグ、それ以外は ⟦n⟧。失敗時は別方式で 1 回だけ再試行
    if (entry.get("entry_kind") or "lang") == "patchouli":
        return ["paired", "bracket"]
    return ["bracket", "ascii"]


# ---------------------------------------------------------------- PFML3 プロンプト
KIND_HINT = {"lang": "In-game UI/item text.", "patchouli": "Guidebook page text.",
             "ftbquests": "Quest book text."}


def build_context(mod_name, namespace, key, kind):
    sentences, ident = [], ""
    mod_name, namespace, key = (mod_name or "").strip(), (namespace or "").strip(), (key or "").strip()
    if mod_name and not unsafe(mod_name):
        ident = f'Minecraft mod "{mod_name}"'
    if namespace and not unsafe(namespace):
        ident += f" (namespace: {namespace})" if ident else f"Minecraft mod namespace: {namespace}"
    if ident:
        sentences.append(ident + ".")
    if key and not unsafe(key):
        sentences.append(f"Translation key: {key}.")
    sentences.append(KIND_HINT.get(kind, KIND_HINT["lang"]))
    return " ".join(sentences)


def contains_term(text, term):
    t, n = text.lower(), term.lower()
    if not n.isascii():
        return n in t  # 日本語などは部分一致でよい
    return re.search(r"(?<![A-Za-z0-9_])" + re.escape(n) + r"(?![A-Za-z0-9_])", t) is not None


def select_glossary(source, glossary):
    """原文に現れる語だけを、長い語から順に返す。"""
    hits = []
    for g in glossary:
        s, t = g["source"].strip(), g["target"].strip()
        if s and t and not unsafe(s) and not unsafe(t) and contains_term(source, s):
            hits.append((s, t))
    hits.sort(key=lambda st: -len(st[0]))
    return hits


def lang_attr(code):
    return f"{KEY}lang{VAL}{code}" if code else ""


def build_prompt(text, source_lang, target_lang, context, glossary):
    # BOS は llama-server が自動で付けるので入れない
    p = f"{TAG}dataset{MSG}translation"
    if context and not unsafe(context):
        p += f"{TAG}context{MSG}{context}"
    if glossary:
        p += f"{TAG}glossary" + "".join(f"{INDEX}{s}{ITEM}{t}" for s, t in glossary)
    p += f"{TAG}input{lang_attr(source_lang)}{MSG}{text}"
    p += f"{TAG}output{lang_attr(target_lang)}{MSG}"
    return p


# ---------------------------------------------------------------- 出力の後処理
def clean_output(text):
    """停止トークンの残りを落とす。--special 起動時は EOS の表記が末尾に付くことがある。"""
    text = text.strip()
    while True:
        new = re.sub(r"<\|[^|<>]*\|>\s*$", "", text).rstrip()
        if new == text:
            break
        text = new
    return text


QUOTE_PAIRS = [("「", "」"), ("『", "』"), ('"', '"'), ("“", "”")]


def strip_wrapping_quotes(text, source):
    """原文が引用文でないのに出力全体を引用符で包んだ場合だけ、外側 1 組を外す。"""
    src = source.lstrip()
    if src[:1] in {o for o, _ in QUOTE_PAIRS} | {"'", "‘"}:
        return text
    t = text.strip()
    for o, c in QUOTE_PAIRS:
        if len(t) > 2 and t.startswith(o) and t.endswith(c) and c not in t[1:-1]:
            return t[1:-1]
    return text


# ---------------------------------------------------------------- llama-server
def log(*a):
    print(time.strftime("%H:%M:%S"), *a, file=sys.stderr, flush=True)


def llama_post(path, body, timeout):
    req = urllib.request.Request(LLAMA_URL + path, json.dumps(body).encode(),
                                 {"Content-Type": "application/json"})
    with urllib.request.urlopen(req, timeout=max(1.0, timeout)) as r:
        return json.load(r)


_slot_ctx = None
_slot_lock = threading.Lock()


def slot_ctx():
    global _slot_ctx
    with _slot_lock:
        if _slot_ctx is None:
            try:
                with urllib.request.urlopen(LLAMA_URL + "/props", timeout=10) as r:
                    _slot_ctx = int(json.load(r)["default_generation_settings"]["n_ctx"])
            except Exception:  # noqa: BLE001
                _slot_ctx = 0  # 取れなければ長さ検査を省く
        return _slot_ctx


def n_tokens(text, add_special, timeout):
    return len(llama_post("/tokenize", {"content": text, "add_special": add_special}, timeout)["tokens"])


def complete(prompt, n_predict, timeout):
    body = {"prompt": prompt, "n_predict": n_predict, "temperature": TEMPERATURE,
            "stop": STOP_STRINGS, "cache_prompt": True}
    return llama_post("/completion", body, timeout)


# ---------------------------------------------------------------- 1 件の翻訳
class EntryFailed(Exception):
    pass


def translate_entry(e, batch, deadline):
    """成功なら訳文、失敗なら EntryFailed(理由)。"""
    source = e.get("source") or ""
    target = plamo_lang(e.get("target_locale"))
    if target is None:
        raise EntryFailed("unsupported_target_language")
    src_lang = plamo_lang(e.get("source_locale") or "en_us")
    if src_lang is None:
        raise EntryFailed("unsupported_source_language")
    if not source.strip():
        return source
    # マスク前の原文で検査する（$(<|plamo:tag|>) のように保護区間へ隠す手を防ぐ）
    if unsafe(source):
        raise EntryFailed("unsafe_source_text")

    kind = e.get("entry_kind") or "lang"
    regex = span_regex(e)
    context = build_context(batch.get("mod_name"), batch.get("namespace"), e.get("key"), kind)
    glossary = select_glossary(source, batch["glossary"])

    last = "format_mismatch"
    for strategy in strategies_for(e):
        masked, spans, roles, openers = mask(source, regex, strategy)
        # 記号だけの文字列は訳すものが無い
        if not re.search(r"\w", MARKER_RE[strategy].sub("", masked)):
            return source
        prompt = build_prompt(masked, src_lang, target, context, glossary)
        remaining = lambda: min(TIMEOUT, deadline - time.monotonic())  # noqa: E731
        if remaining() <= 1:
            raise EntryFailed("batch_deadline")
        n_predict = min(MAX_TOKENS, max(MIN_OUTPUT_TOKENS, 2 * n_tokens(masked, False, remaining()) + 24))
        ctx = slot_ctx()
        if ctx and n_tokens(prompt, True, remaining()) + n_predict > ctx:
            raise EntryFailed("prompt_too_long")
        res = complete(prompt, n_predict, remaining())
        if res.get("truncated") or res.get("stop_type") == "limit":
            raise EntryFailed("output_limit")  # 途中で切れた訳は返さない
        out = strip_wrapping_quotes(clean_output(res.get("content") or ""), source)
        if not out.strip():
            raise EntryFailed("empty_output")
        if unsafe(out):
            raise EntryFailed("special_token_in_output")  # 次の要素を書き始めた等
        try:
            return restore(out, source, regex, strategy, spans, roles, openers)
        except Mismatch as ex:
            last = f"format_mismatch({strategy}: {ex})"
            log(f"[RETRY] id={e.get('id')} {last} out={out!r}")
    raise EntryFailed(last)


# ---------------------------------------------------------------- バッチ
EXECUTOR = ThreadPoolExecutor(PARALLEL)  # 複数バッチが同時に来ても llama-server のスロット数を超えない


def message_text(content):
    if isinstance(content, str):
        return content
    if isinstance(content, list):
        return "".join(p.get("text", "") for p in content if isinstance(p, dict) and p.get("type") == "text")
    return str(content)


def to_glossary(g):
    out = []
    for item in g or []:
        if isinstance(item, dict):
            s = item.get("source") or item.get("term") or item.get("src")
            t = item.get("target") or item.get("translation") or item.get("dst")
            if isinstance(s, str) and isinstance(t, str) and s and t:
                out.append({"source": s, "target": t})
    return out


def wrap(req, content):
    return {
        "id": f"chatcmpl-plamo-{int(time.time() * 1000)}",
        "object": "chat.completion",
        "created": int(time.time()),
        "model": req.get("model", PLAMO_MODEL),
        "choices": [{"index": 0, "message": {"role": "assistant", "content": content}, "finish_reason": "stop"}],
        "usage": {"prompt_tokens": 0, "completion_tokens": 0, "total_tokens": 0},
    }


def handle(req):
    started = time.monotonic()
    deadline = started + BATCH_DEADLINE
    last = message_text(req["messages"][-1]["content"])
    try:
        batch = json.loads(last)
        entries = batch["entries"]
    except Exception:  # バッチ形式でなければ日本語へ単発翻訳（手動テスト用）
        log("[PLAIN] non-batch request")
        e = {"id": 0, "source": last, "source_locale": "en_us", "target_locale": "ja_jp"}
        try:
            return wrap(req, translate_entry(e, {"glossary": []}, deadline)), {}
        except EntryFailed as ex:
            return wrap(req, ""), {"0": str(ex)}
    batch["glossary"] = to_glossary(batch.get("glossary"))

    futures = {EXECUTOR.submit(translate_entry, e, batch, deadline): e for e in entries}
    wait(futures, timeout=max(0.0, deadline - time.monotonic()) + 5)
    out, failures = [], {}
    for f, e in futures.items():
        eid = e.get("id")
        if not f.done():
            f.cancel()  # 未着手なら取り消す。実行中のものは自分の timeout で止まる
            failures[str(eid)] = "batch_deadline"
            continue
        try:
            out.append({"id": eid, "translation": f.result()})
        except EntryFailed as ex:
            failures[str(eid)] = str(ex)
        except Exception as ex:  # noqa: BLE001  通信エラー・タイムアウト等
            failures[str(eid)] = f"error: {type(ex).__name__}: {ex}"
    for eid, why in failures.items():
        log(f"[SKIP] id={eid} {why}")
    log(f"[BATCH] {len(out)}/{len(entries)} ok in {time.monotonic() - started:.1f}s")
    # 失敗した項目は返さない → アプリ側で「応答なし」になり、未翻訳のまま残る（再実行で拾える）
    return wrap(req, json.dumps({"translations": out}, ensure_ascii=False)), failures


def record(req, res, code, failures):
    entry = {"time": datetime.datetime.now().isoformat(timespec="seconds"), "status": code,
             "request": req, "response": res, "failures": failures}
    with open(LOG_FILE, "a", encoding="utf-8") as f:
        f.write(json.dumps(entry, ensure_ascii=False) + "\n")


class H(BaseHTTPRequestHandler):
    def send_json(self, code, obj):
        data = json.dumps(obj, ensure_ascii=False).encode()
        self.send_response(code)
        self.send_header("Content-Type", "application/json; charset=utf-8")
        self.send_header("Content-Length", str(len(data)))
        self.send_header("Access-Control-Allow-Origin", "*")
        self.end_headers()
        self.wfile.write(data)

    def authed(self):
        if API_KEY and self.headers.get("Authorization", "") != f"Bearer {API_KEY}":
            self.send_json(401, {"error": {"message": "Invalid API key", "type": "invalid_request_error"}})
            return False
        return True

    def do_OPTIONS(self):
        self.send_response(204)
        self.send_header("Access-Control-Allow-Origin", "*")
        self.send_header("Access-Control-Allow-Headers", "*")
        self.send_header("Access-Control-Allow-Methods", "GET, POST, OPTIONS")
        self.end_headers()

    def do_GET(self):
        if not self.authed():
            return
        if self.path.rstrip("/").endswith("/models"):
            self.send_json(200, {"object": "list", "data": [
                {"id": PLAMO_MODEL, "object": "model", "created": 0, "owned_by": "pfnet"}]})
        else:
            self.send_json(200, {"status": "ok"})

    def do_POST(self):
        if not self.authed():
            return
        req, failures = None, {}
        try:
            req = json.loads(self.rfile.read(int(self.headers["Content-Length"])))
            if req.get("stream"):
                raise ValueError("stream is not supported")
            (res, failures), code = handle(req), 200
        except Exception as ex:  # noqa: BLE001
            log(f"[BAD REQUEST] {ex}")
            res, code = {"error": {"message": str(ex)}}, 400
        try:
            record(req, res, code, failures)
        except Exception as ex:  # noqa: BLE001
            log(f"[LOG ERROR] {ex}")
        self.send_json(code, res)

    def log_message(self, *a):
        pass


if __name__ == "__main__":
    log(f"listening :{LISTEN_PORT} -> {LLAMA_URL} (parallel={PARALLEL}, deadline={BATCH_DEADLINE}s)")
    ThreadingHTTPServer(("127.0.0.1", LISTEN_PORT), H).serve_forever()


In [ ]:
#@title ③ プロキシ起動 ＋ Cloudflare Tunnel で公開
import subprocess, re, time, threading, os
penv = dict(os.environ, API_KEY=API_KEY, PARALLEL=str(PARALLEL), MAX_TOKENS=str(MAX_TOKENS),
            TEMPERATURE=str(TEMPERATURE), REQUEST_TIMEOUT=str(REQUEST_TIMEOUT),
            BATCH_DEADLINE=str(BATCH_DEADLINE), MODEL_NAME=MODEL_NAME)
if 'proxy' in globals() and proxy.poll() is None:  # 再実行時は再起動
    proxy.terminate(); proxy.wait()
proxy = subprocess.Popen(["python", "/content/plamo_proxy.py"], env=penv,
                         stdout=open("/content/proxy.log", "w"), stderr=subprocess.STDOUT)
time.sleep(2)

tun = subprocess.Popen(["cloudflared", "tunnel", "--no-autoupdate", "--url", "http://127.0.0.1:8100"],
                       stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
PUBLIC_URL = None
for line in tun.stdout:
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
    if m:
        PUBLIC_URL = m.group(0); break
threading.Thread(target=lambda: [None for _ in tun.stdout], daemon=True).start()
time.sleep(5)

print("=" * 60)
print("アプリの「OpenAI 互換（カスタム）」に入力してください")
print("接続先URL:", PUBLIC_URL + "/v1")
print("APIキー  :", API_KEY)
print("モデル   :", MODEL_NAME)
print("=" * 60)

In [ ]:
#@title ④ 動作テスト（任意）
import json, urllib.request
h = {"Authorization": f"Bearer {API_KEY}", "Content-Type": "application/json"}
def post(body):
    r = urllib.request.Request(PUBLIC_URL + "/v1/chat/completions", json.dumps(body).encode(), h)
    return json.load(urllib.request.urlopen(r, timeout=300))["choices"][0]["message"]["content"]

print(json.load(urllib.request.urlopen(urllib.request.Request(PUBLIC_URL + "/v1/models", headers=h))))
# YakuCraft 1.3.0 が送るのと同じ形
def E(i, key, source, kind="lang", ph=()):
    return {"id": i, "key": key, "source": source, "source_locale": "en_us", "target_locale": "ja_jp",
            "entry_kind": kind, "placeholders": list(ph)}
batch = {"mod_name": "test", "namespace": "test",
         "glossary": [{"source": "Diamond", "target": "ダイヤモンド"}],
         "entries": [
             E(1, "item.test.sword", "Diamond Sword"),
             E(2, "msg.test", "%s killed %s with §cfire", ph=["%s", "%s"]),
             E(3, "test.book.page", "Use $(item)heat$() to smelt $(l)ores$().", kind="patchouli")]}
print(post({"model": MODEL_NAME, "messages": [
    {"role": "system", "content": "(YakuCraft の指示文。プロキシでは使わない)"},
    {"role": "user", "content": json.dumps(batch)}]}))
print("失敗理由:", json.loads(open("/content/requests.jsonl", encoding="utf-8").read().splitlines()[-1]).get("failures"))


In [ ]:
#@title ⑤ 記録・ログを見る
import json, os
N = 10  #@param {type:"integer"}
#@markdown 全件は左のファイル欄の `requests.jsonl` を右クリック → ダウンロード
if os.path.exists("/content/requests.jsonl"):
    for l in open("/content/requests.jsonl", encoding="utf-8").read().splitlines()[-N:]:
        e = json.loads(l)
        print("=" * 60)
        print(e["time"], e["status"])
        for m in (e["request"] or {}).get("messages", []):
            print(f"  [{m.get('role')}] {m.get('content')}")
        try:
            print("  [reply]", e["response"]["choices"][0]["message"]["content"])
        except Exception:
            print("  [reply]", e["response"])
        if e.get("failures"):
            print("  [failures]", e["failures"])
else:
    print("まだ記録はありません")
print("\n--- proxy.log（末尾）---")
print(open("/content/proxy.log", encoding="utf-8").read()[-3000:])